### Import Libraries

In [1]:
import pandas as pd
import numpy as np

### Load Dataset

In [45]:
df = pd.read_excel("Dataset for Data Analytics Project 1.xlsx")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 1200
Columns: 14


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


### Initial Inspection
#### Finding Missing Values & Duplicate Rows

In [46]:
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

#### Standarize Column Names

In [47]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

df.columns

Index(['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice',
       'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber',
       'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice'],
      dtype='object')

#### Remove Leading/Trailing Spaces from Text Columns

In [48]:
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

### Missing Value Handling

#### Numerical Columns → Median

In [49]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

#### Categorical Columns → Mode

In [50]:
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)

### Data Type Corrections
### Fix Date Format

In [51]:
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

#### Check invalid dates:

In [52]:
print(df["Date"].isnull().sum())

0


#### Fill invalid dates using median date:

In [53]:
median_date = df["Date"].dropna().median()

df["Date"] = df["Date"].fillna(median_date)

#### Standard format:

In [54]:
df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

### Fix Numeric Columns

In [55]:
numeric_columns = [
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "TotalPrice"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

#### Fill any newly created null values:

In [56]:
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

### Standardize Text Formatting

In [57]:
text_columns = [
    "Product",
    "PaymentMethod",
    "OrderStatus",
    "ReferralSource"
]

for col in text_columns:
    df[col] = df[col].str.title()

### Duplicate Row Handling

In [58]:
duplicate_rows = df.duplicated().sum()

print("Duplicate Rows Found:", duplicate_rows)

df = df.drop_duplicates()

print("Remaining Duplicates:", df.duplicated().sum())

Duplicate Rows Found: 0
Remaining Duplicates: 0


### Duplicate ID Handling

In [59]:
df["OrderID"].duplicated().sum()

np.int64(0)

#### Verification to find Duplicate IDs

In [60]:
print(
    "Duplicate IDs:",
    df["OrderID"].duplicated().sum()
)

Duplicate IDs: 0


### Date Validation

In [61]:
date_check = pd.to_datetime(
    df["Date"],
    errors="coerce"
).isnull().sum()

print("Incorrect Dates:", date_check)

Incorrect Dates: 0


### TotalPrice Validation

In [62]:
calculated_total = (
    df["Quantity"] *
    df["UnitPrice"]
)

incorrect_total = (
    df["TotalPrice"] != calculated_total
).sum()

print("Incorrect Total Prices:", incorrect_total)

Incorrect Total Prices: 107


#### Correction

In [63]:
df["TotalPrice"] = (
    df["Quantity"] *
    df["UnitPrice"]
)

## Export Clean Dataset

In [64]:
df.to_excel(
    "Cleaned_Task_1_Dataset.xlsx",
    index=False
)

print("Dataset Saved Successfully")

Dataset Saved Successfully


## Final Quality Check Report

In [65]:
print("Total Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())
print("Duplicate IDs:", df["OrderID"].duplicated().sum())

date_check = pd.to_datetime(
    df["Date"],
    errors="coerce"
).isnull().sum()

print("Incorrect Dates:", date_check)

Total Missing Values: 0
Duplicate Rows: 0
Duplicate IDs: 0
Incorrect Dates: 0
